In [11]:
model = "llama3.2:1b"

#### Task 1: Simple Chain with Retrieval

**Objective:**

Implement a simple RAG chain with ChatOllama, HuggingFaceEmbeddings and Chroma. 

Process: 

1. Retrieve documents from chroma db based on query
2. Invoke chain with retrieved documents as input

**Task Description:**

- load llm model via ollama
- load embedding model via ollama with `ollama pull pull bge-m3` (if not yet done)
- create chroma db client
- create prompt template for summarization
- create simple chain with following steps: retrieved documents, prompt, model, output parser
- create query and perform similarity search with a query
- invoke chain and pass retrieved documents to the chain


**Useful links:**

- [RAG with Ollama](https://python.langchain.com/v0.2/docs/tutorials/local_rag/)
- [Streaming in Langchain](https://python.langchain.com/docs/concepts/streaming/)


In [ ]:
from langchain_ollama import ChatOllama

# ADD HERE YOUR CODE
model = ChatOllama(model=model)

In [13]:
from langchain_ollama import OllamaEmbeddings

# ADD HERE YOUR CODE
embedding_model = OllamaEmbeddings(model="bge-m3")

In [ ]:
from langchain_chroma import Chroma
import chromadb
import chromadb
from chromadb.config import DEFAULT_TENANT, DEFAULT_DATABASE, Settings

client = chromadb.HttpClient(
    host="localhost",
    port=8000,
    ssl=False,
    headers=None,
    settings=Settings(allow_reset=True, anonymized_telemetry=False),
    tenant=DEFAULT_TENANT,
    database=DEFAULT_DATABASE,
)

# Create a collection
# ADD HERE YOUR CODE
collection = client.get_or_create_collection("ai_model_book")

# Create chromadb
# ADD HERE YOUR CODE
vector_db_from_client = Chroma(client=client,
    collection_name="ai_model_book",
    embedding_function=embedding_model,)

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    "Summarize the main themes in these retrieved docs: {docs}"
)


# Convert loaded documents into strings by concatenating their content
# and ignoring metadata
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


chain = {"docs": format_docs} | prompt | model | StrOutputParser()

In [16]:
search_query = "Types of Machine Learning Systems"

# ADD HERE YOUR CODE
# Perform vector search
docs = vector_db_from_client.similarity_search(search_query, k=3)

print(docs)
formatted_docs_input = {"docs": format_docs(docs)}

[Document(metadata={'page': 33, 'source': './AI_Book.pdf'}, page_content='Types of Machine Learning Systems\nThere are so many different types of Machine Learning systems that it is useful to\nclassify them in broad categories based on:\nWhether or not they are trained with human supervision (supervised, unsuper\nvised, semisupervised, and Reinforcement Learning)\nWhether or not they can learn incrementally on the fly (online versus batch\nlearning)\nWhether they work by simply comparing new data points to known data points,\nor instead detect patterns in the training data and build a predictive model, much\nlike scientists do (instance-based versus model-based learning)\nThese criteria are not exclusive; you can combine them in any way you like. For\nexample, a state-of-the-art spam filter may learn on the fly using a deep neural net\nwork model trained using examples of spam and ham; this makes it an online, model-\nbased, supervised learning system.\nLets look at each of these crite

In [17]:
chain.invoke(docs)

"Based on the retrieved documents, the main themes in Machine Learning systems can be summarized as follows:\n\n**Classification Criteria**\n\n1. **Training data**: Whether or not Machine Learning systems are trained with human supervision (e.g., labeled data) versus without.\n2. **Learning strategy**: Whether they learn incrementally on the fly (batch learning) versus online (supervised, unsupervised, semisupervised).\n3. **Pattern detection**: How the system detects patterns in training data and builds a predictive model.\n\n**Main Themes**\n\n1. **Supervision and Learning**: The importance of supervision during training for effective Machine Learning.\n2. **Generalization**: The ability of a Machine Learning system to perform well on new, unseen examples, whether through instance-based or model-based learning.\n3. **Types of Machine Learning Systems**: Classification into various categories based on the criteria mentioned above.\n\n**Key Concepts**\n\n1. Instance-based vs Model-base

In [18]:
# Simple stream the chain output
for chunk in chain.stream(docs):
    print(chunk, end="", flush=True)

Based on the retrieved documents, here are the main themes:

1. **Classification of Machine Learning Systems**: The documents classify Machine Learning systems into four major categories:
	* Supervised learning (e.g., classification, regression)
	* Unsupervised learning (e.g., clustering, dimensionality reduction)
	* Semisupervised learning
	* Reinforcement learning

2. **Supervision and Monitoring**: The importance of supervising Machine Learning systems is highlighted, particularly for supervised learning where labels are used to train the algorithm. Proper monitoring and switching off or reverting to a previous state if performance drops are also emphasized.

3. **Generalization vs. Instance-based vs. Model-based Learning**: Two main approaches to Machine Learning tasks are discussed: instance-based learning (e.g., labeling individual instances) and model-based learning (e.g., using learned models for new, unseen instances).

4. **Risk Management**: The documents emphasize the need 

In [19]:
# More complex async event streaming
async for event in chain.astream_events(docs, version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)

C:\Users\chiar\AppData\Local\Temp\ipykernel_13444\1927557568.py:2: LangChainBetaWarning: This API is in beta and may change in the future.
  async for event in chain.astream_events(docs, version="v2"):


Based on the retrieved documents, the main themes in Machine Learning Systems can be summarized as follows:

1. **Classification and Categorization**: Machine Learning systems can be classified into four major categories based on supervised, unsupervised, semisupervised, and Reinforcement learning.
2. **Types of Supervised Learning**: There are two types of supervised learning: instance-based (or model-based) and model-based learning. Instance-based learning involves learning from examples by heart, while model-based learning involves learning to predict outcomes using a model.
3. **Machine Learning Approaches**: Machine Learning systems can be categorized based on their generalization capabilities:
	* Instance-based learning: learns from examples and generalizes to new cases through similarity measurement.
	* Model-based learning: learns from data and predicts outcomes without needing additional training.
4. **Criteria for Classification**: The classification criteria include the amou

#### Task 2: Q&A with RAG

**Objective:**

Implement a Q/A retrieval chain with ChatOllama, HuggingFaceEmbeddings and Chroma

**Task Description:**

- create RAG-Q/A prompt template
- create retriever from vector db client (instead of manually passing in docs, we automatically retrieve them from our vector store based on the user question)
- create simple chain with following steps: retriever, formatting retrieved docs, user question, prompt, model, output parser
- create question for Q/A retrieval chain
- invoke chain and with question

**Useful links:**

- [RAG with Ollama](https://python.langchain.com/v0.2/docs/tutorials/local_rag/)

In [ ]:
from langchain_core.runnables import RunnablePassthrough

prompt_template = """
You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.

<context>
{context}
</context>

Answer the following question:

{question}"""


# ADD HERE YOUR CODE
rag_prompt = ChatPromptTemplate.from_template(prompt_template)

# ADD HERE YOUR CODE
retriever = vector_db_from_client.as_retriever(search_kwargs={"k": 3})

# ADD HERE YOUR CODE
qa_rag_chain = ({"context": retriever | format_docs, "question": RunnablePassthrough()} | rag_prompt | model | StrOutputParser())

In [21]:
qa_rag_chain

{
  context: VectorStoreRetriever(tags=['Chroma', 'OllamaEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000001B47D2ED350>, search_kwargs={'k': 3})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], template="\nYou are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\n\n<context>\n{context}\n</context>\n\nAnswer the following question:\n\n{question}"))])
| ChatOllama(model='llama3.2:1b', _client=<ollama._client.Client object at 0x000001B47D038F10>, _async_client=<ollama._client.AsyncClient object at 0x000001B47B7C07D0>)
| StrOutputParser()

In [22]:
question = "What is supervised learning?"

# ADD HERE YOUR CODE
qa_rag_chain.invoke(question)

'Supervised learning is a type of machine learning where the algorithm receives labeled data, which includes desired solutions or outcomes that it needs to learn from. This allows the algorithm to learn patterns and relationships in the data without additional supervision during training. The goal of supervised learning is to make predictions or decisions based on the learned patterns.'

In [23]:
# More complex async event streaming
async for event in qa_rag_chain.astream_events(question, version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)

Supervised learning is a type of machine learning where the algorithm receives labeled data, which includes desired solutions or outcomes. This means that the data has been pre-processed and annotated with the correct answers or results, allowing the algorithm to learn from the information and improve its performance over time.

#### Alternative: Using pre-built ConversationalRetrievalChain Class

In [24]:
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory

In [25]:
retriever = vector_db_from_client.as_retriever()
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

In [26]:
qa_chain = ConversationalRetrievalChain.from_llm(
    model, retriever=retriever, memory=memory, verbose=False
)

In [27]:
# More complex async event streaming
async for event in qa_chain.astream_events("What is supervised learning?", version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)

Supervised learning is a type of machine learning where an algorithm is trained on labeled data, meaning that the training data includes examples with corresponding labels or outputs.

In supervised learning, you have:

1. **Training data**: A set of examples, often in the form of inputs (x) and outputs (y), where x represents the feature(s) to be predicted, and y represents the actual output.
2. **Labels or annotations**: Each example is labeled with a target value or class label that corresponds to its expected output.

The goal of supervised learning is to develop a model that can predict the output (y) for new, unseen examples in the training data based on their input features (x).

Here's an example:

Suppose you want to build a system to classify images of cats and dogs as either "cat" or "dog". You have a labeled dataset with thousands of images, where each image is annotated with its category.

In supervised learning, you would:

1. Collect the training data (e.g., 10,000 image

In [28]:
# More complex async event streaming
async for event in qa_chain.astream_events("Which algorithms can be used there?", version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)

Here is the rephrased standalone question based on the original conversation:

What type of machine learning algorithm can be used for unsupervised learning, particularly for tasks such as pattern discovery or structure identification in unlabeled data?I don't know. The original text doesn't mention specific types of machine learning algorithms that are commonly used for unsupervised learning, particularly for tasks like pattern discovery or structure identification in unlabeled data. It does mention several algorithms that can be used for unsupervised learning, such as autoencoders and restricted Boltzmann machines, but it doesn't provide a comprehensive overview of the types of machine learning algorithms that are suitable for these tasks.